# Phase 1: MVP, a coding agent that can fix a bug safely

Phase 0's agent can only **read**. It can explain a bug but not fix it. This phase gives it the tools to explore, edit and run code, with safety checks before anything risky.

See [phase_1_README.md](phase_1_README.md) for the explanation behind each section.

## Goal

**Done when:** given a small project with a failing test, the agent

1. finds the bug,
2. fixes it,
3. runs the tests to confirm the fix,
4. and asks before anything risky.

## Plan

One section per roadmap step, built strictly in this order. Each step starts from what already exists and changes only what it needs.

| # | Section | What it adds | Why at this point |
|---|---|---|---|
| 1 | Tool registry | One place that lists the tools; the prompt, parser and loop read from it | Set up before adding tools, so each new tool is a function, not a prompt rewrite |
| 2 | List and find files | Tools to see what's in a project | The agent has to explore a project before it can change it |
| 3 | Folder boundary | File tools refuse paths outside the project | Before any tool can write |
| 4 | Write and edit files | Tools that change files | The moment the agent can actually change code |
| 5 | Approval prompt | Ask the user before risky actions | Needed before shell access |
| 6 | Run shell commands | A tool that runs commands | Lets the agent run code, tests and git |
| 7 | Verify loop | Run the tests, fix what fails, repeat | What makes it a *coding* agent |

After section 7: an end-to-end demo against the goal above. Then this notebook's code is refactored into `agent_v1/` (see [WORKFLOW.md](../WORKFLOW.md)).

## How this notebook works

- It imports **only** Phase 0's frozen package, `agent_v0`. Nothing comes from `agent_v1`, which doesn't exist yet.
- Everything new is built here. To change a Phase 0 function, we look at the imported version, then write a new one **with the same name** in this notebook (*shadowing*). We never modify `agent_v0` itself.

## 0. Setup: import Phase 0

`phase-0-foundation` has a hyphen in its name, so Python can't import it as a package. Instead we add the folder to Python's search path, then import `agent_v0` from inside it.

In [ ]:
import sys

sys.path.insert(0, "../phase-0-foundation")

from agent_v0 import SYSTEM_PROMPT, ask, call_llm, new_conversation, parse_tool_call, read_file

A quick check that it's the Phase 0 agent we know: its prompt still describes one hand-written tool.

In [ ]:
print(SYSTEM_PROMPT)

## 1. Tool registry: game plan

*Draft for review. Nothing below is built yet; we agree on this plan first.*

### What we're doing and why

Phase 0 has one tool, `read_file`, and its name is written by hand in **three places** in `agent_v0/loop.py`:

| Place | Lines | What's hard-coded |
|---|---|---|
| The system prompt | [`loop.py:10-14`](../phase-0-foundation/agent_v0/loop.py) | "You have ONE tool: read_file(filename) ..." |
| The parser | [`loop.py:22`](../phase-0-foundation/agent_v0/loop.py) | `prefix = "tool: read_file("` |
| The loop | [`loop.py:51-52`](../phase-0-foundation/agent_v0/loop.py) | `result = read_file(args.get("filename", ""))` |

So adding a second tool would mean editing all three. The roadmap's goal for this step ([`ROADMAP.md:181`](../ROADMAP.md)): *"Describe tools automatically from their code; add a tool without touching the loop."* It comes first in Phase 1 because steps 2, 4 and 6 all add tools ([`ROADMAP.md:70`](../ROADMAP.md)).

### Sub-steps

Each one is a separate turn: built, run and discussed before the next.

| # | Sub-step | Replaces |
|---|---|---|
| 1a | **Look at the problem.** Print Phase 0's prompt, parser and `ask()`, and find the three hard-coded places in the table above. | — |
| 1b | **The registry.** One place that lists the tools: a dictionary from tool name to function. Register Phase 0's `read_file`, unchanged. | — |
| 1c | **Describe tools from the registry.** Build the prompt's tool list from what's registered, instead of typing it. | `loop.py:10-14` |
| 1d | **Parse any tool name.** The parser accepts `tool: <name>({...})` and returns which tool was asked for, plus its arguments. | `loop.py:22` |
| 1e | **Run a tool by name.** Look it up in the registry and call it. An unknown tool or wrong arguments come back as an error result, not a crash. | `loop.py:51-52` |
| 1f | **The new loop.** Phase 0's `ask()`, rewritten in this notebook to use 1c–1e. | `ask()` |
| 1g | **Try it.** Phase 0's demo question, then print the conversation. | — |

### Evidence: how real agents do it

Source code read on 2026-10-31:

| Agent | How it registers tools | Source |
|---|---|---|
| **learn-claude-code** (lesson s02) | Two separate pieces: a hand-written list of tool descriptions (`TOOLS`) and a dictionary from name to function (`TOOL_HANDLERS`). The loop calls `TOOL_HANDLERS[name](**args)`; an unknown name returns `"Unknown: <name>"`. Its own summary: *"the loop stays the same; only tool registration and dispatch grow."* | [`s02_tool_use/code.py:138-156, 180-181`](https://github.com/shareAI-lab/learn-claude-code/blob/HEAD/s02_tool_use/code.py#L138-L181) |
| **ghuntley/how-to-build-a-coding-agent** (Go) | One bundle per tool, `ToolDefinition`: name, description, input schema and function together. The input schema is **generated** from a type, not hand-written. The loop searches the tool list by name; an unknown name returns "tool not found". | [`read.go:253-271, 147-170`](https://github.com/ghuntley/how-to-build-a-coding-agent/blob/HEAD/read.go#L253-L271) |
| **mini-swe-agent** (text mode) | **No registry.** It has one action type, a shell command, pulled out of the reply with a regex. A badly formatted reply is sent back to the model as an error message. | [`actions_text.py:15-40`](https://github.com/SWE-agent/mini-swe-agent/blob/HEAD/src/minisweagent/models/utils/actions_text.py#L15-L40) |

What they agree on, and so what this plan follows:
- **A dictionary (or list) from tool name to function**, and the loop looks tools up by name (1b, 1e).
- **An unknown tool is an error result the model sees**, not a crash (1e). This also matches Phase 0's own rule: `read_file` returns errors instead of raising them.

Two differences to keep in mind:
- Both learn-claude-code and ghuntley use **native tool calling** (the API receives the tool list and returns structured requests). We keep our text protocol for now; native tool calling is step 12, and comparing the two is step 13 ([`ROADMAP.md:98-99`](../ROADMAP.md)).
- mini-swe-agent shows the opposite design: no registry, one shell tool. The roadmap lists this as "Bash-only agent", scoped Later ([`ROADMAP.md:187`](../ROADMAP.md)).

### Decisions for you

These are design choices, not settled by the evidence. Each is my judgment until you decide.

1. **Where does a tool's description come from?** (affects 1b, 1c)
   - *Hand-written, next to the registry*, like learn-claude-code: full control over the wording, but code and description can drift apart.
   - *Generated from the function itself*, from its name, parameter names and docstring: closest to the roadmap's "describe tools automatically from their code", and to ghuntley generating the schema. The description can only be as good as the docstring.
2. **Should the prompt show a concrete example line** (as Phase 0 does, `loop.py:14`) **or only a generic pattern?** (affects 1c) A concrete example is probably followed more reliably. In the code I wrote too early today (since removed), a generic pattern let the model send a bare string instead of `{"filename": ...}` in 1 of 4 test runs. That's one small observation, not a measurement.
3. **What if the arguments aren't a JSON object?** (affects 1d, 1e) Reject them in the parser with a clear error, like mini-swe-agent's format errors, or let 1e's "bad arguments" error catch it.

### Done when

- With only `read_file` registered, the agent answers Phase 0's question the same way as before.
- Adding a tool needs only registering a function: no change to the prompt, parser or loop.
- Asking for an unknown tool, or passing wrong arguments, gives the model an error result instead of crashing.
